In [ ]:
# Post-process URBANopt (non-connected) results — shared helpers live in lib.helpers.
from pathlib import Path

from urbanopt_des.urbanopt_analysis import URBANoptAnalysis

from lib.helpers import (
    apply_default_style,
    plot_combined_monthly_boxplots,
    plot_end_use_bar_non_connected,
    plot_grid_metric_boxplots,
    plot_load_duration_curves,
    plot_seasonal_boxplots,
    silence_common_warnings,
)

apply_default_style()
silence_common_warnings()

# URBANopt Post-Process (Non-Connected)

This notebook post-processes a **URBANopt-only** scenario (no DES/Modelica required).

What this notebook does:
1. Configures a project/scenario path and validates required files.
2. Builds `uo_analysis` and processes URBANopt outputs.
3. Computes grid and summary metrics.
4. Generates and saves a few standard plots and CSV summaries.

What this notebook does not require:
- Modelica result files (`*.mat`)
- DES post-processing inputs

In [ ]:
# Step 1: point at either a URBANopt project directory (contains `run/`) or a
# scenario directory directly under `run/`. Examples:
#   ./esbe_2026/activity_05/coincident
#   ./esbe_2026/activity_05/coincident/run/baseline_scenario
MODEL_OUTPUT_SUBDIR = "../test_activities/esbe_2026"
input_path = (
    Path(f"{MODEL_OUTPUT_SUBDIR}/activity_05/coincident").expanduser().resolve()
)
year = 2017

uo_analysis, paths = URBANoptAnalysis.bootstrap_from_uo_results(
    input_path,
    scenario_name=None,
    year_of_data=year,
    display_name="Non-Connected",
)

# Pull out commonly-used paths for downstream cells
uo_project_dir = paths["uo_project_dir"]
scenario_name = paths["scenario_name"]
scenario_results_dir = paths["scenario_results_dir"]
results_summary_dir = paths["results_summary_dir"]
project_geojson_filename = paths["geojson_path"]

print("=== Configuration (review before continuing) ===")
print(f"Input path: {input_path}")
print(f"URBANopt project dir: {uo_project_dir}")
print(f"Scenario: {scenario_name}")
print(f"GeoJSON: {project_geojson_filename}")
print(f"Results dir: {scenario_results_dir}")
print(f"Results summary dir: {results_summary_dir}")

## Processing Step

This next cell creates the URBANopt analysis object and computes all core non-connected outputs.

It will:
- load URBANopt scenario results,
- build aggregations and summaries,
- run the non-connected-safe conversion calls,
- and keep the resulting dataframes in memory for plotting.

In [ ]:
# Run the rest of the URBANopt post-process: variables, combined frames,
# resampling, aggregations, rollups, building summaries. No Modelica results
# here so the modelica path is a no-op.
uo_analysis.resample_and_convert_modelica_results([], [])
uo_analysis.save_modelica_variables()
uo_analysis.save_urbanopt_results_in_modelica_paths()
uo_analysis.combine_modelica_and_openstudio_results()
uo_analysis.resample_actual_data()
uo_analysis.save_dataframes("min_60_with_buildings")
uo_analysis.create_modelica_aggregations()
uo_analysis.create_rollups()
uo_analysis.create_building_summaries()
uo_analysis

## Metrics Step

This cell computes grid metrics, summary tables, and building-level annual metrics.

Outputs are written to the scenario output directory so they can be reused by other notebooks or reports.

In [ ]:
# Grid metrics + summary results + per-building rollups.
uo_analysis.calculate_all_grid_metrics()
uo_analysis.save_dataframes(["grid_metrics_daily", "grid_metrics_annual"])

uo_analysis.create_summary_results()
uo_analysis.save_dataframes(["grid_summary", "end_use_summary"])
analysis_name_mappings = uo_analysis.display_name_mappings()

buildings_df = uo_analysis.create_building_level_results()
buildings_df.to_csv(
    uo_analysis.urbanopt.scenario_output_path / "building_metrics_annual.csv",
    index=True,
)
print(
    f"Saved to {uo_analysis.urbanopt.scenario_output_path / 'building_metrics_annual.csv'}"
)
display(buildings_df)

### Simple Graphs

In [ ]:
# Bar plot of the Non-Connected end-use summary (also writes the CSV).
plot_end_use_bar_non_connected(uo_analysis, results_summary_dir)

In [ ]:
# Per-analysis monthly + weekday energy box plots.
plot_seasonal_boxplots(uo_analysis, results_summary_dir)

In [ ]:
# Combined monthly box plots (overlay every analysis as a hue group).
plot_combined_monthly_boxplots(uo_analysis, results_summary_dir)

In [ ]:
# Daily grid metric box plots per analysis.
plot_grid_metric_boxplots(uo_analysis, results_summary_dir)

In [ ]:
# Load duration curves (full + zoomed first-100 hours).
# Note: Total Natural Gas does not include the DES boiler yet.
plot_load_duration_curves(uo_analysis, results_summary_dir)